# Great Expectations

Una misma suite de GX 1.x, tres orígenes de datos:

| Origen | Dónde se calcula | Método |
|---|---|---|
| DataFrame en memoria | En la Function (pandas) | `ctx.gx.validate(df, ...)` |
| Archivo del datalake | En la Function (pandas), después de leer | `ctx.datalake.read_*` + `ctx.gx.validate` |
| Tabla o vista de Synapse | **En Synapse**: GX traduce las expectativas a SQL | `ctx.gx.validate_query(..., modo='sql')` |

Para tablas grandes use Synapse: los datos no viajan a la Function. Más contexto en
[03_synapse.ipynb](03_synapse.ipynb) y [MOTOR_DE_COMPUTO.md](MOTOR_DE_COMPUTO.md).

In [ ]:
import logging
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# Raíz del repo (el notebook vive en docs/) y variables locales: .env y .env.synapse (ignorados por git).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT))
for env_file in ('.env', '.env.synapse'):
    load_dotenv(ROOT / env_file, override=True)

# Solo en el notebook: ejecutar tareas que no están en el YAML. En la Function las tareas viven en el YAML.
os.environ['TAREA_DEBE_CONFIGURACION'] = 'false'

from centraal_dataframework.runner import reset_runner  # noqa: E402
from centraal_dataframework.settings import clear_settings_cache  # noqa: E402

clear_settings_cache()  # relee las variables recién cargadas
reset_runner()
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

HAY_STORAGE = bool(os.environ.get('AZURE_STORAGE_CONNECTION_STRING') or os.environ.get('STORAGE_ACCOUNT_NAME'))

# Sin storage, GX guarda los reportes en una carpeta local temporal y todo el notebook funciona.
# Con AZURE_STORAGE_CONNECTION_STRING (en .env), los reportes van al contenedor CONTENEDOR_VALIDACIONES
# y resultado.url es un link con SAS (SAS_VALOR_EXPIRACION / SAS_UNIDAD_EXPIRACION).
# Sección 4 (datalake): DATALAKE_TEST_PATH=contenedor/carpeta en .env (ver docs/datalake.env.example).
RUTA = os.environ.get('DATALAKE_TEST_PATH', '').strip().strip('/')
# Sección 5 (Synapse): SYNAPSE_WORKSPACE en .env.synapse (ver docs/synapse.env.example), el extra [synapse]
# y el driver ODBC. Opcional: SYNAPSE_TEST_GX_QUERY / SYNAPSE_TEST_GX_COLUMN para una tabla real.
HAY_SYNAPSE = bool(os.environ.get('SYNAPSE_WORKSPACE') or os.environ.get('SYNAPSE_SERVER'))
GX_QUERY = os.environ.get('SYNAPSE_TEST_GX_QUERY', '').strip()
GX_COLUMN = os.environ.get('SYNAPSE_TEST_GX_COLUMN', '').strip()

print('storage:', 'sí' if HAY_STORAGE else 'no (reportes locales)')
print('datalake (sección 4):', RUTA or 'SALTADA: falta DATALAKE_TEST_PATH')
print('synapse (sección 5):', 'sí' if HAY_SYNAPSE else 'SALTADA: falta SYNAPSE_WORKSPACE')

## 1. Validar un DataFrame (pandas en memoria)

In [ ]:
import great_expectations as gx
import pandas as pd

from centraal_dataframework import TaskContext, task

ventas = pd.DataFrame({'canal': ['web', 'tienda', 'web'], 'valor': [100.0, 250.5, 80.0]})

# Expectativas GX 1.x: clases de gx.expectations.
EXPECTATIVAS = [
    gx.expectations.ExpectColumnValuesToNotBeNull(column='canal'),
    gx.expectations.ExpectColumnValuesToBeInSet(column='canal', value_set=['web', 'tienda']),
    gx.expectations.ExpectColumnValuesToBeBetween(column='valor', min_value=0),
    gx.expectations.ExpectTableRowCountToBeBetween(min_value=1),
]


@task(uses=['gx'])
def validar_ventas(ctx: TaskContext):
    """Devuelve el DQResult; el runner decide qué hacer si falla."""
    # En una tarea real los datos se leen aquí (ctx.datalake / ctx.synapse). Aquí llegan por params.
    return ctx.gx.validate(ctx.params['df'], 'ventas', EXPECTATIVAS)


resultado = validar_ventas(params={'df': ventas})
print('success:', resultado.success, '| fallidas:', resultado.failed)
print('reporte:', resultado.url)

## 2. Fallo de calidad

In [ ]:
from centraal_dataframework import ErrorTareaCalidadDatos
from centraal_dataframework.runner import get_runner

malas = ventas.assign(valor=[100.0, -5.0, 80.0])  # un valor negativo

# Llamada directa: devuelve el DQResult fallido sin lanzar (para explorar).
resultado = validar_ventas(params={'df': malas})
print('success:', resultado.success, '| fallidas:', resultado.failed)

# Por el runner (como en la Function) el fallo es ErrorTareaCalidadDatos y se notifica (aquí notify=False).
# En la Function se notifica en el primer intento, no se reintenta y las dependientes no se ejecutan.
try:
    get_runner().run_task('validar_ventas', params={'df': malas}, notify=False)
except ErrorTareaCalidadDatos as error:
    print('error de calidad:', error.failed)

## 3. Silenciar una alarma conocida

In [ ]:
import tempfile
from datetime import date

from centraal_dataframework.config import load_framework_config, silence_warnings
from centraal_dataframework.runner import Runner

# En el YAML de la Function. hasta (incluido) y motivo son obligatorios: no hay silencios permanentes.
YAML = """
tareas:
  validar_ventas:
    silenciar_dq:
      hasta: 2099-12-31
      motivo: "Proveedor X envía valores negativos, ticket #123"
"""
yaml_path = Path(tempfile.mkdtemp()) / 'centraal_dataframework.yaml'
yaml_path.write_text(YAML, encoding='utf-8')

runner = Runner(conf=load_framework_config(yaml_path))
runner.add_task(validar_ventas, 'validar_ventas')

# Silencio vigente: no lanza ni notifica, deja un WARNING en el log y devuelve el DQResult.
# La tarea queda exitosa y sus dependientes se ejecutan. Al vencer, la alarma vuelve sola.
print('success:', runner.run_task('validar_ventas', params={'df': malas}).success)

# `validate-config` avisa los silencios vencidos y los que vencen en los próximos 7 días.
print(silence_warnings(runner.conf, date(2099, 12, 28)))

## 4. Origen datalake: leer a pandas y validar

La tarea lee el archivo con `ctx.datalake` (solo las columnas que valida) y valida el DataFrame en la Function.
Sirve para archivos pequeños: pandas carga todo el archivo en memoria.

In [ ]:
import uuid


@task(uses=['datalake', 'gx'])
def validar_archivo_ventas(ctx: TaskContext):
    """Valida el archivo del lote; en la Function el YAML define params.ruta y la programación."""
    df = ctx.datalake.read_csv(ctx.params['ruta'], sep='|', usecols=['canal', 'valor'])
    return ctx.gx.validate(df, 'ventas_archivo', EXPECTATIVAS)


# YAML en la Function:
#   tareas:
#     validar_archivo_ventas:
#       horas: '7'
#       params: {ruta: contenedor/ventas/ventas.csv}
if HAY_STORAGE and RUTA:
    from centraal_dataframework.resources import get_datalake

    datalake = get_datalake()
    base = f'{RUTA}/cdf_gx_notebook_{uuid.uuid4().hex[:8]}'
    try:
        datalake.write_csv(ventas, f'{base}/ventas.csv', sep='|', index=False)
        resultado = validar_archivo_ventas(params={'ruta': f'{base}/ventas.csv'})
        print('datalake -> success:', resultado.success, '| fallidas:', resultado.failed)
    finally:
        datalake.fs.rm(base, recursive=True)  # limpieza
else:
    print('SALTADO: configure el storage y DATALAKE_TEST_PATH en .env')

## 5. Origen Synapse: el cómputo lo hace Synapse

`ctx.gx.validate_query` recibe un SELECT y la **misma suite**:

- `modo='sql'` (plan A, por defecto): GX traduce cada expectativa a SQL, Synapse la ejecuta y a la
  Function solo vuelven conteos. No se traen filas.
- `modo='pandas'` (plan B): trae el resultado con `read_df` y valida en la Function. Úselo con una
  expectativa que serverless no acepte, o con un dataset pequeño que escanea mucho.

El ejemplo usa una tabla en línea (`VALUES`): solo necesita el workspace configurado. Con una tabla
real, pida **solo las columnas que valida** y filtre la partición con un literal
(`WHERE fecha = '{{fecha}}'`, con `params={'fecha': ...}`) para que serverless pode carpetas.

In [ ]:
# Las mismas filas del ejemplo de la sección 1, ahora calculadas en Synapse.
QUERY_VENTAS = """
SELECT canal, valor
FROM (VALUES ('web', 100.0), ('tienda', 250.5), ('web', 80.0)) AS v(canal, valor)
"""


@task(uses=['synapse', 'gx'])
def validar_ventas_synapse(ctx: TaskContext):
    """La suite corre en Synapse (modo sql) o en la Function (modo pandas), según params.modo."""
    return ctx.gx.validate_query(
        ctx.synapse,
        name='ventas_synapse',
        query=ctx.params.get('query', QUERY_VENTAS),
        params=ctx.params.get('params'),  # valores de {{param}}, con lista blanca
        expectations=ctx.params.get('expectativas', EXPECTATIVAS),
        modo=ctx.params.get('modo', 'sql'),
        medir_costo=True,  # MB procesados del día antes y después (aproximado)
    )


if HAY_SYNAPSE:
    for modo in ('sql', 'pandas'):
        r = validar_ventas_synapse(params={'modo': modo})
        print(f'{modo:6} -> success: {r.success} | fallidas: {r.failed} | MB procesados: {r.data_processed_mb}')
else:
    print('SALTADO: configure SYNAPSE_WORKSPACE en .env.synapse')

In [ ]:
# Fallo de calidad calculado en Synapse: un valor negativo viola ExpectColumnValuesToBeBetween(min_value=0).
QUERY_MALAS = QUERY_VENTAS.replace('80.0', '-5.0')

if HAY_SYNAPSE:
    r = validar_ventas_synapse(params={'query': QUERY_MALAS})
    print('sql -> success:', r.success, '| fallidas:', r.failed)

# Con una tabla real (SYNAPSE_TEST_GX_QUERY, p.ej. un SELECT de una vista de bronze con WHERE fecha = '...').
if HAY_SYNAPSE and GX_QUERY and GX_COLUMN:
    basicas = [
        gx.expectations.ExpectTableRowCountToBeBetween(min_value=1),
        gx.expectations.ExpectColumnValuesToNotBeNull(column=GX_COLUMN, mostly=0.95),
    ]
    r = validar_ventas_synapse(params={'query': GX_QUERY, 'expectativas': basicas})
    print('tabla real -> success:', r.success, '| fallidas:', r.failed, '| MB procesados:', r.data_processed_mb)

## 6. Cuál usar

| Situación | Uso |
|---|---|
| El dato ya está en memoria (respuesta de una API, un Excel pequeño) | `ctx.gx.validate(df, ...)` |
| Archivo pequeño en el lago | `ctx.datalake.read_*` (solo las columnas necesarias) + `ctx.gx.validate` |
| Tabla o vista en Synapse, de cualquier tamaño | `ctx.gx.validate_query(..., modo='sql')` |
| La consulta de Synapse escanea mucho, o una expectativa falla en serverless | Misma llamada con `modo='pandas'`, sobre una partición pequeña |

Los fallos y los silencios (secciones 2 y 3) funcionan igual con los tres orígenes.